# jiu-jitsu-auto-scoring
*This notebook is a **WIP**!*


In [1]:
# prompt: mount drive and clone the repository in the current branch, the organization is called jiu-jitsu-auto-scoring and the repository is called model
from google.colab import drive

BRANCH = "8-fine-tune-vitpose-on-harmony4d" # Write the name of the branch you're working in here

!git clone -q https://github.com/jiu-jitsu-auto-scoring/model.git
%cd /content/model
!git pull
!git checkout $BRANCH

fatal: destination path 'model' already exists and is not an empty directory.
/content/model
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.52 KiB | 1.52 MiB/s, done.
From https://github.com/jiu-jitsu-auto-scoring/model
   2ea85f4..596040f  8-fine-tune-vitpose-on-harmony4d -> origin/8-fine-tune-vitpose-on-harmony4d
Updating 2ea85f4..596040f
Fast-forward
 src/Fine_Tune_VitPose_Harmony4d.ipynb | 99 +++++++++++++++++++++++++++++++++++++++++++++++++++++--
 1 file changed, 97 insertions(+), 2 deletions(-)
Already on '8-fine-tune-vitpose-on-harmony4d'
Your branch is up to date with 'origin/8-fine-tune-vitpose-on-harmony4d'.


#0. Imports, Requirements, etc.
Upload the required checkpoint into that folder

In [3]:
# Import drive for custom images
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install dependencies

In [5]:
# Step 0: Uninstall existing torch-related packages to avoid conflicts
!pip uninstall -y torch torchvision torchaudio

# Step 1: Install basic Python packages (datasets, transformers, etc.)
!pip install datasets transformers huggingface_hub wandb matplotlib scikit-learn opencv-python

# Step 2: Install a complete compatible PyTorch stack with CUDA support
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

# Step 3: Install OpenMIM (OpenMMLab package manager)
!pip install -U openmim

# Step 4: Install MMEngine with a specific version known to work with PyTorch 2.0.1
!pip install mmengine==0.7.4

# Step 5: Install MMCV using the exact pre-built wheel for PyTorch 2.0.1
!pip install mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html

# Step 6: Install MMPose with a compatible version
!pip install mmpose==1.1.0

# Step 7: Install any additional dependencies that might be needed
!pip install xtcocotools

Found existing installation: torch 2.0.1
Uninstalling torch-2.0.1:
  Successfully uninstalled torch-2.0.1
Found existing installation: torchvision 0.15.2
Uninstalling torchvision-0.15.2:
  Successfully uninstalled torchvision-0.15.2
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 673.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 92.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: mmengine
    Found existing installation: mmengine 0.10.7
    Uninstalling mmengine-0.10.7:
      Successfully uninstalled mmengine-0.10.7
Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html


Choose the datasets you wanna download from Harmony4d: https://huggingface.co/datasets/Jyun-Ting/Harmony4D/tree/main/train

In [3]:
import os
import glob
import numpy as np
import json
import cv2
import matplotlib.pyplot as plt
import random
from pathlib import Path

def prepare_harmony4d_for_vitpose(base_dir="harmony4d_data"):
    """
    Collect and prepare Harmony4D data for ViTPose training
    """
    print("Preparing Harmony4D dataset for ViTPose training...")
    all_data = []

    # Find all subdirectories with number_activity format (e.g., 001_hugging)
    activity_dirs = glob.glob(f"{base_dir}/*_*/")

    for activity_dir in activity_dirs:
        activity_name = os.path.basename(os.path.normpath(activity_dir))
        print(f"Processing {activity_name}...")

        # Process exo folder (contains camera views)
        exo_dir = os.path.join(activity_dir, "exo")
        if not os.path.exists(exo_dir):
            print(f"  No exo directory found in {activity_dir}, skipping.")
            continue

        # Get all camera directories
        camera_dirs = glob.glob(os.path.join(exo_dir, "cam*"))

        for camera_dir in camera_dirs:
            camera_name = os.path.basename(camera_dir)
            print(f"  Processing {camera_name}...")

            # Get image directory for this camera
            image_dir = os.path.join(camera_dir, "images")
            if not os.path.exists(image_dir):
                print(f"    No images directory found in {camera_dir}, skipping.")
                continue

            # Get all images
            image_files = glob.glob(os.path.join(image_dir, "*.jpg"))
            print(f"    Found {len(image_files)} images")

            # Get corresponding poses2d and bbox directories
            poses2d_dir = os.path.join(activity_dir, "processed_data", "poses2d", camera_name)
            bbox_dir = os.path.join(activity_dir, "processed_data", "bbox", camera_name)

            if not os.path.exists(poses2d_dir):
                print(f"    No poses2d directory found for {camera_name}, skipping.")
                continue

            # Process each image
            for image_path in image_files:
                image_basename = os.path.splitext(os.path.basename(image_path))[0]

                # Find corresponding files
                poses2d_path = os.path.join(poses2d_dir, f"{image_basename}.npy")
                bbox_path = os.path.join(bbox_dir, f"{image_basename}.npy")

                if not os.path.exists(poses2d_path) or not os.path.exists(bbox_path):
                    continue

                try:
                    # Load pose and bbox data
                    poses2d_data = np.load(poses2d_path, allow_pickle=True)
                    if poses2d_data.ndim == 0:  # Handle 0-D array
                        poses2d_data = poses2d_data.item()

                    bbox_data = np.load(bbox_path, allow_pickle=True)
                    if bbox_data.ndim == 0:  # Handle 0-D array
                        bbox_data = bbox_data.item()

                    # Process each person in the image
                    for person_id in poses2d_data.keys():
                        if person_id in bbox_data:
                            keypoints = poses2d_data[person_id]
                            bbox = bbox_data[person_id]

                            # Store the data
                            data_item = {
                                "image_path": image_path,
                                "person_id": person_id,
                                "keypoints": keypoints,
                                "bbox": bbox,
                                "activity": activity_name,
                                "camera": camera_name
                            }
                            all_data.append(data_item)
                except Exception as e:
                    print(f"    Error processing {image_path}: {e}")

    print(f"Total samples collected: {len(all_data)}")
    return all_data

def convert_to_coco_format(data, output_path, relative_path_base=None):
    """
    Convert the collected data to COCO format for ViTPose training
    """
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{
            "id": 1,
            "name": "person",
            "supercategory": "person",
            "keypoints": [
                "nose", "left_eye", "right_eye", "left_ear", "right_ear",
                "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
                "left_wrist", "right_wrist", "left_hip", "right_hip",
                "left_knee", "right_knee", "left_ankle", "right_ankle"
            ],
            "skeleton": [
                [16, 14], [14, 12], [17, 15], [15, 13], [12, 13], [6, 12],
                [7, 13], [6, 7], [6, 8], [7, 9], [8, 10], [9, 11], [2, 3],
                [1, 2], [1, 3], [2, 4], [3, 5], [4, 6], [5, 7]
            ]
        }]
    }

    image_id_map = {}
    annotation_id = 1

    for idx, item in enumerate(data):
        image_path = item["image_path"]

        # Get or create image ID
        if image_path not in image_id_map:
            # Get image dimensions
            img = cv2.imread(image_path)
            height, width = img.shape[:2]

            image_id = len(image_id_map) + 1
            image_id_map[image_path] = image_id

            # Create relative path if base is provided
            if relative_path_base:
                file_name = os.path.relpath(image_path, relative_path_base)
            else:
                file_name = image_path

            # Add image info
            coco_data["images"].append({
                "id": image_id,
                "file_name": file_name,
                "width": width,
                "height": height
            })
        else:
            image_id = image_id_map[image_path]

        # Format keypoints
        keypoints = item["keypoints"]
        # COCO format requires flattened [x1, y1, v1, x2, y2, v2, ...] format
        flattened_keypoints = []
        for kp in keypoints:
            if len(kp) == 3:  # [x, y, visibility]
                flattened_keypoints.extend(kp)
            elif len(kp) == 2:  # [x, y]
                flattened_keypoints.extend([kp[0], kp[1], 2])  # Assuming visible

        # Format bbox (COCO uses [x, y, width, height])
        x_min, y_min, x_max, y_max = item["bbox"]
        width = x_max - x_min
        height = y_max - y_min
        coco_bbox = [x_min, y_min, width, height]

        # Add annotation
        coco_data["annotations"].append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": 1,
            "bbox": coco_bbox,
            "area": width * height,
            "segmentation": [],
            "iscrowd": 0,
            "keypoints": flattened_keypoints,
            "num_keypoints": len(keypoints)
        })

        annotation_id += 1

    # Save to JSON file
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(coco_data, f)

    print(f"COCO format data saved to {output_path}")
    print(f"Total images: {len(coco_data['images'])}")
    print(f"Total annotations: {len(coco_data['annotations'])}")

    return coco_data

def visualize_samples(data, num_samples=5):
    """
    Visualize random samples from the dataset to verify keypoints and bounding boxes
    """
    if not data:
        print("No data available for visualization")
        return

    # Select random samples
    sample_indices = random.sample(range(len(data)), min(num_samples, len(data)))
    samples = [data[i] for i in sample_indices]

    # Create figure
    fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
    if len(samples) == 1:
        axes = [axes]

    for i, sample in enumerate(samples):
        # Load image
        img = cv2.imread(sample["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Draw bounding box
        x_min, y_min, x_max, y_max = sample["bbox"]
        cv2.rectangle(img, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 255, 0), 2)

        # Draw keypoints
        colors = [(255, 0, 0), (0, 0, 255), (255, 255, 0)]  # Different colors for visibility
        for kp_idx, kp in enumerate(sample["keypoints"]):
            if len(kp) == 3:  # [x, y, visibility]
                x, y, v = kp
                if v > 0:  # Only draw visible keypoints
                    color_idx = int(v) - 1 if int(v) < len(colors) else 0
                    cv2.circle(img, (int(x), int(y)), 4, colors[color_idx], -1)
            elif len(kp) == 2:  # [x, y]
                x, y = kp
                cv2.circle(img, (int(x), int(y)), 4, colors[0], -1)

        # Display
        axes[i].imshow(img)
        axes[i].set_title(f"{sample['activity']} - {sample['camera']}")
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

def main():
    base_dir = "harmony4d_data"  # Directory with extracted data
    output_dir = "vitpose_data"
    os.makedirs(output_dir, exist_ok=True)

    # Collect and process data
    harmony4d_data = prepare_harmony4d_for_vitpose(base_dir)

    # Visualize samples to verify data
    visualize_samples(harmony4d_data)

    # Split data into train and validation sets (80/20)
    random.shuffle(harmony4d_data)
    split_idx = int(len(harmony4d_data) * 0.8)
    train_data = harmony4d_data[:split_idx]
    val_data = harmony4d_data[split_idx:]

    # Convert to COCO format for ViTPose
    train_coco = convert_to_coco_format(
        train_data,
        os.path.join(output_dir, "harmony4d_train.json"),
        relative_path_base=base_dir
    )

    val_coco = convert_to_coco_format(
        val_data,
        os.path.join(output_dir, "harmony4d_val.json"),
        relative_path_base=base_dir
    )

    print("Dataset preparation completed!")
    print(f"Training samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    print(f"Data saved to: {output_dir}")

    return {
        "train_json": os.path.join(output_dir, "harmony4d_train.json"),
        "val_json": os.path.join(output_dir, "harmony4d_val.json"),
        "train_samples": len(train_data),
        "val_samples": len(val_data)
    }

# Run the data preparation
dataset_info = main()

Preparing Harmony4D dataset for ViTPose training...
Total samples collected: 0
No data available for visualization
COCO format data saved to vitpose_data/harmony4d_train.json
Total images: 0
Total annotations: 0
COCO format data saved to vitpose_data/harmony4d_val.json
Total images: 0
Total annotations: 0
Dataset preparation completed!
Training samples: 0
Validation samples: 0
Data saved to: vitpose_data
